In [1]:
# 학습된 모델 대신 직접 생성한 embedding을 사용(toy-embedding)
import math
import re
import pandas as pd
import torch
import torch.nn.functional as F

In [2]:
sentences = [
    '이 영화 정말 좋다',
    '이 영화 너무 지루하다',
    '배우 연기가 감동적이다',
    '전개는 보통이지만 마지막이 재미있다'
]

In [3]:
# toy-embedding : 2차원 벡터를 임의로 생성
# positive단어는 오른쪽 위 쪽에 배치
# negative 단어는 왼쪽 아래쪽에 배치
# neutal 단어는 원점 근처에 배치

# query와 비슷한 방향의 단어가 더 큰 score를 받음
toy_vocab = {
    '이': torch.tensor([0.0, 0.0]),
    '영화': torch.tensor([0.1, 0.0]),
    '정말': torch.tensor([0.0, 0.2]),
    '좋다': torch.tensor([1.2, 1.0]),
    '너무': torch.tensor([0.0, 0.1]),
    '지루하다': torch.tensor([-1.2, -1.0]),
    '배우': torch.tensor([0.2, 0.1]),
    '연기가': torch.tensor([0.3, 0.1]),
    '감동적이다': torch.tensor([1.0, 1.2]),
    '전개는': torch.tensor([0.0, 0.0]),
    '보통이지만': torch.tensor([0.0, 0.0]),
    '마지막이': torch.tensor([0.0, 0.1]),
    '재미있다': torch.tensor([1.1, 1.1]),
    '별로다': torch.tensor([-1.0, -1.1]),
    '최악이다': torch.tensor([-1.3, -1.2]),
}

poistive_query = torch.tensor([1.0,1.0])
negative_query = torch.tensor([-1.0,-1.0])
toy_vocab['좋다'], poistive_query, negative_query

(tensor([1.2000, 1.0000]), tensor([1., 1.]), tensor([-1., -1.]))

## 전처리 함수 만들기
- 문장을 토큰으로나눈뒤, 각 토큰을 toy embedding 벡터로 바꾼다
- 여기서 key value는 같은 벡터를 사용, query는 지금 찾고싶은 의미

In [ ]:
def simple_tokenize(text: str):
    text = re.sub(r'[^가-힣\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    if not text:
        return []
    return text.split()

def get_token_vectors(tokens, vocab):
    vectors = []
    for token in tokens:
        vectors.append(vocab.get(token, torch.tensor([0.0, 0.0])))
    return torch.stack(vectors) if vectors else torch.empty(0, 2)

def attention_with_query(tokens, query, vocab):
    key = get_token_vectors(tokens, vocab)
    value = key

    if key.numel() == 0:
        return pd.DataFrame(), torch.tensor([])

    d_k = key.size(-1)
    scores = torch.matmul(key, query) / math.sqrt(d_k)
    weights = F.softmax(scores, dim=0)
    context = torch.sum(weights.unsqueeze(-1) * value, dim=0)

    frame = pd.DataFrame({
        'token': tokens,
        'key_x': key[:, 0].tolist(),
        'key_y': key[:, 1].tolist(),
        'score': scores.tolist(),
        'weight': weights.tolist(),
    })
    frame['value_x'] = frame['key_x']
    frame['value_y'] = frame['key_y']
    return frame, context